In [119]:
import pandas as pd
import pytz
import re

# many sheets
anySheet = pd.ExcelFile("chatbot_phase2.xlsx") # ['ยังไม่แยกอาหาร this', 'แยกส่วนประกอบมาแล้ว', 'Total']
sheetName = anySheet.sheet_names

# get a specific sheet
df = pd.read_excel("chatbot_phase2.xlsx", sheet_name=sheetName[1])


In [120]:
df["Entry"].nunique()

2371

In [121]:
df["Entry"].value_counts()

Entry
943    2
978    2
945    2
985    2
950    2
      ..
25     1
27     1
28     1
32     1
34     1
Name: count, Length: 2371, dtype: int64

In [122]:
#Actual Bloating

df['date&time'] = pd.to_datetime(df['date&time'], dayfirst=True, errors='coerce')
df = df.sort_values(by=['user_id', 'date&time']).reset_index(drop=True)

df['date&time'] = pd.to_datetime(df['date&time'], dayfirst=True, errors='coerce')
df = df.sort_values(by=['user_id', 'date&time']).reset_index(drop=True)

def get_meal_sequences_t1_t2(group):
    group["Actual Bloating (t-1,t-2)"] = 0   # new column, default = 0
    #t-1 and t-2
    for pos, (i, row) in enumerate(group.iterrows()):
        if row['ynbloating'] == 1:
            for j in range(max(0, pos-2), pos):
                group.at[group.index[j], "Actual Bloating (t-1,t-2)"] = 1

    return group

def get_meal_sequences_t1(group):
    group = group.copy()
    group["Actual Bloating (t-1)"] = 0
    
    for pos, (i, row) in enumerate(group.iterrows()):
        if row['ynbloating'] == 1 and pos > 0:
            group.at[group.index[pos-1], "Actual Bloating (t-1)"] = 1

    return group

def get_meal_sequences_t2(group):
    group = group.copy()
    group["Actual Bloating (t-2)"] = 0
    
    for pos, (i, row) in enumerate(group.iterrows()):
        if row['ynbloating'] == 1 and pos > 0:
            group.at[group.index[pos-2], "Actual Bloating (t-2)"] = 1

    return group
    
# group by user
result = df.groupby('user_id').apply(get_meal_sequences_t1_t2).reset_index(drop=True)
result = result.groupby('user_id').apply(get_meal_sequences_t1).reset_index(drop=True)
result = result.groupby('user_id').apply(get_meal_sequences_t2).reset_index(drop=True)
result.to_excel("new.xlsx", index=False)


C:\Users\Windows 10 Pro\AppData\Local\Temp\ipykernel_7312\1876978797.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = df.groupby('user_id').apply(get_meal_sequences_t1_t2).reset_index(drop=True)
C:\Users\Windows 10 Pro\AppData\Local\Temp\ipykernel_7312\1876978797.py:41: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = result.groupby('user_id').apply(get_meal_sequences_t1).reset_index(drop=True)
C:\

In [123]:
# Load your file again
df = pd.read_excel("new.xlsx")

# Focus only on List menu column
menus = df["List menu"].dropna()

def clean_menu(text):
    text = str(text)
    # 1. Replace newline with space
    text = text.replace("\n", " ")
    # 2. Replace punctuation with space (like . , ; )
    text = re.sub(r"[.,;]", " ", text)
    # 3. Remove multiple spaces
    text = re.sub(r"\s+", " ", text).strip()
    # 4. Split by space
    items = text.split(" ")
    # 5. Remove empty strings
    items = [i for i in items if i.strip()]
    return items

# Apply cleaning function
df["Cleaned_menu"] = menus.apply(clean_menu)

print(df["Cleaned_menu"])
# df_to_save = df[["date&time","user_name","food_time","ynbloating","Actual Bloating (t-1,t-2)","List menu","Cleaned_menu"]]
# df_to_save.to_excel("newone.xlsx", index=False)


0       [มะระ, หมู, ผักชี, เห็นชอบ, กะปิ, สะตอ, ข้าว, ...
1             [ข้าว, หมู, คะน้า, หอมใหญ่, มะเขือเทศ, ไข่]
2                                 [กาแฟ, ข้าวเหนียว, หมู]
3        [ข้าว, เนื้อวัว, ใบกะเพรา, พริก, กะเพรา, ไข่ไก่]
4       [ข้าว, เนื้อหมู, กะทิ, มะเขือพวง, ใบมะกรูด, ผั...
                              ...                        
2383    [เอ็กเบเนดิก, ไข่, เห็ด, มะเขือเทศ, อโวคาโด, ซ...
2384                      [น้ำมะพร้าว, แก้วมังกร, มังคุด]
2385    [ปลาสำลีทอด, ผักกูดลวก, กระเจี๊ยบเขียว, มะเขือ...
2386             [แป้งข้าวหมาก, มังคุด, กล้วย, แก้วมังกร]
2387                          [โจ๊กไก่, แก้วมังกร, กล้วย]
Name: Cleaned_menu, Length: 2388, dtype: object


In [124]:
#Delete 100% Fodmap ingredient
def remove_words(food_list, remove_list):
    return [item for item in food_list if item not in remove_list]

def remove_substring(food_list, sub):
    return [item.replace(sub, "") for item in food_list if sub not in item or item.replace(sub, "")]

words_to_remove = ['ข้าว', 'หมู',"เนื้อหมู", "ข้าวสวย", "พริก"]
#substring_to_remove = "ข้าว"

# Apply function to the column
df["Cleaned_menu_deleted"] = df["Cleaned_menu"].apply(lambda x: remove_words(x, words_to_remove))
#df["Cleaned_menu_deleted"] = df["Cleaned_menu_deleted"].apply(lambda x: remove_substring(x, substring_to_remove))
print(df["Cleaned_menu_deleted"])

df_to_save = df[["List menu","Cleaned_menu_deleted"]]
#df_to_save.to_excel("newtwo.xlsx", index=False)

0             [มะระ, ผักชี, เห็นชอบ, กะปิ, สะตอ, ผักบุ้ง]
1                        [คะน้า, หอมใหญ่, มะเขือเทศ, ไข่]
2                                      [กาแฟ, ข้าวเหนียว]
3                    [เนื้อวัว, ใบกะเพรา, กะเพรา, ไข่ไก่]
4       [กะทิ, มะเขือพวง, ใบมะกรูด, ผักกาดดอง, เนื้อไก...
                              ...                        
2383    [เอ็กเบเนดิก, ไข่, เห็ด, มะเขือเทศ, อโวคาโด, ซ...
2384                      [น้ำมะพร้าว, แก้วมังกร, มังคุด]
2385    [ปลาสำลีทอด, ผักกูดลวก, กระเจี๊ยบเขียว, มะเขือ...
2386             [แป้งข้าวหมาก, มังคุด, กล้วย, แก้วมังกร]
2387                          [โจ๊กไก่, แก้วมังกร, กล้วย]
Name: Cleaned_menu_deleted, Length: 2388, dtype: object


In [125]:
import re

remove_exact = {
    'ชิ้น','ลูก','ถุง','ขวด','กล่อง','แพ็ค','แพค','ซอง','ถ้วย','แผ่น','หัว','ก้อน','ฝัก','กลีบ',
    'ช้อน','ช้อนโต๊ะ','ช้อนชา','กรัม','กก','กิโล','กิโลกรัม','kg','g','ml','มล','ลิตร','ซีซี',
    'แพ็ก','กระป๋อง','กระปุก','อัน','ชุด','เสิร์ฟ',
    'บันทึก'   
}

stopwords = {'และ','กับ','ใส่','ไม่','ประมาณ','ครึ่ง','เต็ม'}

time_words = {
    "เช้า","ตอนเช้า","เช้าๆ",
    "กลางวัน","ตอนกลางวัน",
    "เย็น","ตอนเย็น",
    "ดึก","ตอนดึก","ก่อนนอน",
    "บ่าย","สาย","ค่ำ","เที่ยง",
    "มื้อ","มื้อเช้า","มื้อกลางวัน","มื้อเย็น"
}

digit_pattern = re.compile(r'[\d๑๒๓๔๕๖๗๘๙๐]')
punct_pattern = re.compile(r'^[\W_]+$')

def is_non_food(token: str) -> bool:
    if not isinstance(token, str):
        return True
    
    # normalize quotes, strip space
    s = token.replace("“", '"').replace("”", '"')
    s = s.replace("‘", "'").replace("’", "'")
    s = s.strip()
    
    # unique quotes
    s = s.strip(" .,:;()[]{}\"'")
    
    #non-food conditions
    if s == "":
        return True
    if digit_pattern.search(s):
        return True
    if s.lower() in remove_exact or s in stopwords or s in time_words:
        return True
    if punct_pattern.match(s):
        return True
    if len(s) <= 1:
        return True
    
    return False

def remove_time_words(s: str) -> str:
    
    for tw in time_words:
        # begining words
        s = re.sub(f'^{tw}', '', s)
        # ending words
        s = re.sub(f'{tw}$', '', s)
    return s.strip()

def clean_list(lst):
    # string → list
    items = eval(lst) if isinstance(lst, str) else lst
    out = []
    for tok in items:
        if not isinstance(tok, str):
            continue
        # token with + / -
        sub_tokens = re.split(r'[+\-/]', tok)
        for s in sub_tokens:
            s = s.strip()
            
            s = re.sub(r'^[\d\.\,\-\(\)]+', '', s)
            s = s.strip(" .,:;()[]{}\"'")
            
            s = remove_time_words(s)
        
            words = re.findall(r'[A-Za-zก-๙]+', s)
            for w in words:
                if w and not is_non_food(w):
                    out.append(w)
    # stay at the same order
    seen, uniq = set(), []
    for x in out:
        if x not in seen:
            uniq.append(x)
            seen.add(x)
    return uniq

df["Food_only_v3"] = df["Cleaned_menu_deleted"].apply(clean_list)
df["Number"] = range(1, len(df) + 1)

df_to_save = df[["List menu","user_id","Number","Food_only_v3"]]
#df_to_save.to_excel("final_regex.xlsx", index = False)

#df_to_save = df[["List menu","Cleaned_menu_deleted","Food_only_v3"]]
#df_to_save.to_excel("try6.xlsx", index=False)


In [126]:
df.to_excel("full3.xlsx", index = False)

In [127]:
df_to_save = df[["user_id","Number", "List menu","Food_only_v3"]]
df_to_save.to_excel("final_regex3.xlsx", index = False)

In [128]:
print(df['user_id'].nunique())

62


In [129]:
print(df['user_id'].nunique())

62


In [130]:
# Bloating Class

freq = df.groupby('user_id')['ynbloating'].mean() * 100
freq = freq.reset_index()
#freq = freq.reset_index().rename(columns={'ynbloating': 'bloating_fre'})
threshold = 80
freq['bloating_class'] = freq['ynbloating'].apply(lambda x: 'High' if x >= threshold else 'Low')
freq = freq.rename(columns={'ynbloating': 'bloating_freq'})
print(freq)

df = df.merge(freq, on='user_id', how='left')
#df.to_excel("try8.xlsx", index=False)

                              user_id  bloating_freq bloating_class
0   U00a33f8c677a22fcfe4cd1d1f426600b     100.000000           High
1   U02a1d5008350a6cd7715d83520710bad      23.255814            Low
2   U07aac43990ab7e5628a6e4f3b4bdf35d     100.000000           High
3   U080d1a058439b61c9ba9e6e99c10a901      38.461538            Low
4   U0a7f789177696d2f6dbb7713abfabb2b      45.833333            Low
..                                ...            ...            ...
57  Uf45171994bda44f6f96d488254d8a64b      11.111111            Low
58  Uf4f53d7ac203a413151bae5525b98035      95.614035           High
59  Uf686100a45dd111ee8a9875a01d82058      26.923077            Low
60  Uf915c13ad880cbd83bd7d949c51b9d63       4.761905            Low
61  Uff35caf0357cd131a143bf44cc012f59      16.666667            Low

[62 rows x 3 columns]


In [131]:
#Filter high and low

df_low = df[df["bloating_class"] == "Low"]

df_high = df[df["bloating_class"] == "High"]


In [132]:
df["user_id"].nunique()

62

In [133]:
number_users=df_low["user_id"].nunique()
print(f"จำนวนคนไม่ที่ sensitive : {number_users} คน")
number_meals = df_low.shape[0]
print(f"จำนวนมื้ออาหารทั้งหมด : {number_meals} มื้อ")

percent = round(df_low["ynbloating"].value_counts()[1]/df_low.shape[0],4) * 100
print(f"percentage จากมื้ออาหารทั้งหมดที่เป็น bloating : {percent}%")
print("===========================")
print(df_low["ynbloating"].value_counts())

จำนวนคนไม่ที่ sensitive : 47 คน
จำนวนมื้ออาหารทั้งหมด : 1498 มื้อ
percentage จากมื้ออาหารทั้งหมดที่เป็น bloating : 29.37%
ynbloating
0    1058
1     440
Name: count, dtype: int64


In [134]:
number_users=df_high["user_id"].nunique()
print(f"จำนวนคนที่ sensitive : {number_users} คน")
number_meals = df_high.shape[0]
print(f"จำนวนมื้ออาหารทั้งหมด : {number_meals} มื้อ")

percent = round(df_high["ynbloating"].value_counts()[1]/df_high.shape[0],4) * 100
print(f"percentage จากมื้ออาหารทั้งหมดที่เป็น : {percent}%")
print("===========================")
print(df_high["ynbloating"].value_counts())


จำนวนคนที่ sensitive : 15 คน
จำนวนมื้ออาหารทั้งหมด : 890 มื้อ
percentage จากมื้ออาหารทั้งหมดที่เป็น : 92.25%
ynbloating
1    821
0     69
Name: count, dtype: int64


In [135]:
asso = df_high
final_data_t1_t2_high = []
for foods, bloating in zip(asso["Food_only_v3"], asso["Actual Bloating (t-1,t-2)"]):
    items = foods.copy()  # copy the list so we don't modify original
    if bloating == 1:
        items.append("bloating")
    final_data_t1_t2_high.append(items)

asso = df_low
final_data_t1_t2_low = []
for foods, bloating in zip(asso["Food_only_v3"], asso["Actual Bloating (t-1,t-2)"]):
    items = foods.copy()  # copy the list so we don't modify original
    if bloating == 1:
        items.append("bloating")
    final_data_t1_t2_low.append(items)



In [136]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

#One-hot encode
te = TransactionEncoder()
te_ary = te.fit(final_data_t1_t2_high).transform(final_data_t1_t2_high)
basket_df = pd.DataFrame(te_ary, columns=te.columns_)

# Frequent itemsets
frequent_itemsets = apriori(basket_df, min_support=0.01, use_colnames=True)

# Association rules, force RHS = bloating
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)
rules = rules[rules['consequents'] == frozenset({'bloating'})]
result_sorted = rules.sort_values(by ="confidence", ascending=False)
print(result_sorted[['antecedents', 'consequents', 'support', 'confidence', "lift"]])
#result_sorted[['antecedents', 'consequents', 'support', 'confidence', 'lift']].to_excel("association_rules_highClass.xlsx", index=False)


     antecedents consequents   support  confidence      lift
0     (กระเทียม)  (bloating)  0.071910    1.000000  1.037296
1        (กล้วย)  (bloating)  0.013483    1.000000  1.037296
2         (กะทิ)  (bloating)  0.050562    1.000000  1.037296
3         (กะปิ)  (bloating)  0.037079    1.000000  1.037296
4    (กะหล่ำปลี)  (bloating)  0.022472    1.000000  1.037296
..           ...         ...       ...         ...       ...
74       (องุ่น)  (bloating)  0.012360    0.846154  0.877712
138  (ปลา, กุ้ง)  (bloating)  0.012360    0.846154  0.877712
58        (มะระ)  (bloating)  0.012360    0.846154  0.877712
30          (นม)  (bloating)  0.011236    0.833333  0.864413
102    (ไส้กรอก)  (bloating)  0.012360    0.785714  0.815018

[160 rows x 5 columns]


In [137]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

#One-hot encode
te = TransactionEncoder()
te_ary = te.fit(final_data_t1_t2_low).transform(final_data_t1_t2_low)
basket_df = pd.DataFrame(te_ary, columns=te.columns_)

# Frequent itemsets
frequent_itemsets = apriori(basket_df, min_support=0.01, use_colnames=True)

# Association rules, force RHS = bloating
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)
rules = rules[rules['consequents'] == frozenset({'bloating'})]
result_sorted = rules.sort_values(by ="confidence", ascending=False)
print(result_sorted[['antecedents', 'consequents', 'support', 'confidence', "lift"]])

   antecedents consequents   support  confidence      lift
4  (มะม่วงสุก)  (bloating)  0.012016    0.782609  1.906257
5    (เต้าหู้)  (bloating)  0.015354    0.676471  1.647728
3    (ข้าวต้ม)  (bloating)  0.012016    0.600000  1.461463


In [138]:
# #LLM
# from langchain.prompts import PromptTemplate
# from langchain_ollama import OllamaLLM

# llm = OllamaLLM(model="llama3", temperature=0.1)
# prompt_template = PromptTemplate(
#     input_variables=["menu_text"],
#     template="""
#     You are a data cleaning assistant.
#     Given the following menu text, return a Python list of food items only.
#     - Remove empty words, punctuation, and irrelevant symbols.
#     - Correct obvious typos if possible.
    
#     Menu: "{menu_text}"
    
#     Return only the cleaned list.
#     """
# )

# def clean_menu_llama(menu_text):
#     # Fill the prompt with the actual menu
#     filled_prompt = prompt_template.format(menu_text=menu_text)
    
#     # Get LLM response
#     response = llm.invoke(filled_prompt)
    
#     # Convert the string output to a Python list
#     try:
#         cleaned_list = eval(response)
#     except:
#         # fallback: split by space
#         cleaned_list = str(menu_text).replace("\n"," ").split()
#     return cleaned_list

# # menu_text = " ".join(df["Cleaned_menu"].iloc[180])  # convert list -> string

# # cleaned_list = clean_menu_llama(menu_text)
# # print(cleaned_list)

# # print(df["Cleaned_menu"].iloc[180])


In [139]:
asso = df


final_data_t1_t2 = []
for foods, bloating in zip(asso["Food_only_v3"], asso["Actual Bloating (t-1,t-2)"]):
    items = foods.copy()  
    if bloating == 1:
        items.append("bloating")
    final_data_t1_t2.append(items)

final_data_t1 = []
for foods, bloating in zip(asso["Food_only_v3"], asso["Actual Bloating (t-1)"]):
    items = foods.copy()  
    if bloating == 1:
        items.append("bloating")
    final_data_t1.append(items)

final_data_t2 = []
for foods, bloating in zip(asso["Food_only_v3"], asso["Actual Bloating (t-2)"]):
    items = foods.copy()  
    if bloating == 1:
        items.append("bloating")
    final_data_t2.append(items)

#print(final_data)


In [140]:
#import ast

#result = [ast.literal_eval(item) for item in converted]
#print(result)



In [141]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

#One-hot encode
te = TransactionEncoder()
te_ary = te.fit(final_data_t1_t2).transform(final_data_t1_t2)
basket_df = pd.DataFrame(te_ary, columns=te.columns_)

# Frequent itemsets
frequent_itemsets = apriori(basket_df, min_support=0.01, use_colnames=True)

# Association rules, force RHS = bloating
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)
rules = rules[rules['consequents'] == frozenset({'bloating'})]
result_sorted = rules.sort_values(by ="confidence", ascending=False)
print(result_sorted[['antecedents', 'consequents', 'support', 'confidence', "lift"]])

#result_sorted[['antecedents', 'consequents', 'support', 'confidence', 'lift']].to_excel("association_rules_dfAll.xlsx", index=False)


       antecedents consequents   support  confidence      lift
31        (แครรอท)  (bloating)  0.010888    1.000000  1.621181
26   (ส้มแมนดาริน)  (bloating)  0.015494    0.973684  1.578519
33     (โกโก้ร้อน)  (bloating)  0.012563    0.967742  1.568885
22         (ยาจีน)  (bloating)  0.010050    0.960000  1.556334
43  (กาแฟ, โปรตีน)  (bloating)  0.011725    0.903226  1.464293
34        (โปรตีน)  (bloating)  0.013400    0.888889  1.441050
20     (มะม่วงสุก)  (bloating)  0.011307    0.843750  1.367872
28        (หอมแดง)  (bloating)  0.021357    0.822581  1.333552
29       (เต้าหู้)  (bloating)  0.023032    0.820896  1.330820
6         (ขนมจีน)  (bloating)  0.013400    0.820513  1.330200
11        (ตะไคร้)  (bloating)  0.010888    0.812500  1.317210
3           (กาแฟ)  (bloating)  0.033082    0.797980  1.293670
9        (ข้าวโพด)  (bloating)  0.012563    0.789474  1.279880
19       (ผักสลัด)  (bloating)  0.011307    0.771429  1.250626
10         (คะน้า)  (bloating)  0.015494    0.770833  1

In [142]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

#One-hot encode
te = TransactionEncoder()
te_ary = te.fit(final_data_t1).transform(final_data_t1)
basket_df = pd.DataFrame(te_ary, columns=te.columns_)

# Frequent itemsets
frequent_itemsets = apriori(basket_df, min_support=0.01, use_colnames=True)

# Association rules, force RHS = bloating
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)
rules = rules[rules['consequents'] == frozenset({'bloating'})]
result_sorted = rules.sort_values(by ="confidence", ascending=False)
print(result_sorted[['antecedents', 'consequents', 'support', 'confidence', "lift"]])

       antecedents consequents   support  confidence      lift
10   (ส้มแมนดาริน)  (bloating)  0.015075    0.947368  1.837787
15     (โกโก้ร้อน)  (bloating)  0.012144    0.935484  1.814732
14        (แครรอท)  (bloating)  0.010050    0.923077  1.790664
22  (กาแฟ, โปรตีน)  (bloating)  0.011725    0.903226  1.752155
16        (โปรตีน)  (bloating)  0.013400    0.888889  1.724343
7         (ตะไคร้)  (bloating)  0.010050    0.750000  1.454915
2           (กาแฟ)  (bloating)  0.030988    0.747475  1.450016
3         (ขนมจีน)  (bloating)  0.012144    0.743590  1.442480
13       (เต้าหู้)  (bloating)  0.020101    0.716418  1.389769
17      (โยเกิร์ต)  (bloating)  0.016750    0.714286  1.385633
1           (กะปิ)  (bloating)  0.013400    0.711111  1.379475
5        (ข้าวโพด)  (bloating)  0.011307    0.710526  1.378340
12        (หอมแดง)  (bloating)  0.018007    0.693548  1.345405
9        (ผักสลัด)  (bloating)  0.010050    0.685714  1.330208
0           (กะทิ)  (bloating)  0.018844    0.681818  1

In [143]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

#One-hot encode
te = TransactionEncoder()
te_ary = te.fit(final_data_t2).transform(final_data_t2)
basket_df = pd.DataFrame(te_ary, columns=te.columns_)

# Frequent itemsets
frequent_itemsets = apriori(basket_df, min_support=0.01, use_colnames=True)

# Association rules, force RHS = bloating
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)
rules = rules[rules['consequents'] == frozenset({'bloating'})]

result_sorted = rules.sort_values(by ="confidence", ascending=False)
print(result_sorted[['antecedents', 'consequents', 'support', 'confidence', "lift"]])

                 antecedents consequents   support  confidence      lift
24  (โกโก้ร้อน, ส้มแมนดาริน)  (bloating)  0.010050    1.000000  1.939886
13               (โกโก้ร้อน)  (bloating)  0.012982    1.000000  1.939886
8              (ส้มแมนดาริน)  (bloating)  0.015494    0.973684  1.888837
12                  (แครรอท)  (bloating)  0.010050    0.923077  1.790664
21            (กาแฟ, โปรตีน)  (bloating)  0.010888    0.838710  1.627001
14                  (โปรตีน)  (bloating)  0.012563    0.833333  1.616572
1                     (กะปิ)  (bloating)  0.014238    0.755556  1.465692
6                   (ตะไคร้)  (bloating)  0.010050    0.750000  1.454915
10                  (หอมแดง)  (bloating)  0.019263    0.741935  1.439270
2                     (กาแฟ)  (bloating)  0.029732    0.717172  1.391232
15                (โยเกิร์ต)  (bloating)  0.016750    0.714286  1.385633
0                     (กะทิ)  (bloating)  0.019682    0.712121  1.381434
9                    (ส้มโอ)  (bloating)  0.012144 